# Module 07 — AI Workbook Companion (GPU neural-network inference)

A lightweight companion to
[cuda_neural_network_demo_complete.ipynb](../cuda_neural_network_demo_complete.ipynb).
It does **not** replace it — it adds the supervision layer from
[Module 11](../../11/README.md) to Module 07's topic: **batched neural-network
inference**, and the kind of silent, shape-preserving math error an AI makes when
vectorising it.

New here? Read [Module 11's README](../../11/README.md) first. The failure mode
here is one from the README list: an **"optimization" or vectorisation that
changes the math**. The output still has the right *shape*, so it slips past a
shape check while being numerically wrong.

## The loop, applied to a classifier's softmax

Your task: turn a batch of logits `(batch, num_classes)` into per-row class
probabilities with a softmax — the last layer of the demo classifier. You may
have an AI write it. Your job is everything around it.

1. **SPECIFY** — Contract: input shape `(batch, classes)`, output shape same,
   the invariant (**each row sums to 1**), and the metric.
2. **GENERATE** — Ask the AI for the batched softmax. Keep it separate.
3. **PREDICT** — *Before running:* the classic vectorisation bug is reducing over
   the **wrong axis** (across the batch instead of across classes). Do you see it?
4. **VERIFY** — Use [verify_softmax.py](./verify_softmax.py): a reference softmax
   with a per-row-sum-to-1 check and an element-wise comparison.
5. **DIAGNOSE** — Explain why the wrong-axis version still ran and still produced
   numbers in [0,1], yet is wrong.

## The adversarial exercise

[adversarial_softmax_buggy.py](./adversarial_softmax_buggy.py) is a softmax "an
AI wrote for you." It runs, returns the right shape, and every value is in
[0,1] — but it normalises **down the batch** instead of **across the classes**,
so the rows do not sum to 1 and the predicted class can be wrong.

Your job:

1. Predict the bug from the axis choice *before* running.
2. Run it and check whether each row sums to 1.
3. State the exact bug (`axis=0` vs `axis=1`, and the missing max-subtraction for
   numerical stability) and the fix.
4. Prove the fix against the reference.

Could an AI catch this? Sometimes — but "the output is the right shape and looks
like probabilities" passes a casual review while the normalisation axis is wrong.
Checking the row sums and comparing to a reference catches it. That is your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Why did a wrong reduction axis still produce values in [0,1] and the right
  shape? What single check exposes it immediately?
- Where does subtracting the per-row max before `exp` matter, and what GPU
  failure (overflow to `inf`) does it prevent?
- On the GPU this softmax is a per-row reduction. What would you check to be sure
  a generated CUDA/CuPy version reduces over the correct dimension?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Run the problem program [adversarial_softmax_buggy.py](./adversarial_softmax_buggy.py). Read it first and predict what will happen before you run this cell.

In [ ]:
!python adversarial_softmax_buggy.py

## Step 2 - Run your verification harness

[verify_softmax.py](./verify_softmax.py) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the function under test. Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!python verify_softmax.py

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.